# Gold Labeling: Wyckoff State-Machine

In [0]:
%run ../config/config

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DoubleType, IntegerType,
)

#### 1. Read inputs

In [0]:
cleaned_df = spark.table(tables["cleaned_ohlcv"])
indicators_df = spark.table(tables["technical_indicators"])
wyckoff_df = spark.table(tables["wyckoff_features"])

# join
joined_df = (
    cleaned_df
    .join(indicators_df, on=["symbol", "date"], how="inner")
    .join(wyckoff_df, on=["symbol", "date"], how="inner")
)
print(f"Joined rows: {joined_df.count():,}")

#### 2. Wyckoff state labeler

In [0]:
label_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("date", DateType(), False),
    StructField("phase_label", IntegerType(), False),
    StructField("phase_name", StringType()),
    StructField("phase_confidence", DoubleType()),
    StructField("tr_support", DoubleType()),
    StructField("tr_resistance", DoubleType()),
    StructField("tr_id", IntegerType()),
])

cfg = WyckoffLabelingConfig()


def detect_prior_trend(close, sma_20, sma_50, idx, cfg, direction):
    """
    "down" for downtrend (accumulation), "up" for uptrend (distribution).
    """
    if idx < cfg.prior_trend_bars:
        return False, 0.0

    start = max(0, idx - cfg.prior_trend_bars)
    count = 0
    for i in range(start, idx):
        if not np.isnan(sma_20[i]) and not np.isnan(sma_50[i]):
            if direction == "down" and sma_20[i] < sma_50[i]:
                count += 1
            elif direction == "up" and sma_20[i] > sma_50[i]:
                count += 1
    sma_trend = count >= cfg.prior_trend_bars * 0.8

    lookback_start = max(0, idx - cfg.prior_trend_lookback)
    if close[lookback_start] > 0:
        if direction == "down":
            move = (close[lookback_start] - close[idx]) / close[lookback_start]
        else:
            move = (close[idx] - close[lookback_start]) / close[lookback_start]
    else:
        move = 0.0
    price_moved = move >= cfg.prior_trend_decline

    if sma_trend and price_moved:
        return True, min(1.0, 0.5 + move)
    elif sma_trend or price_moved:
        return True, 0.5
    return False, 0.0


def find_selling_climax(high, low, close, volume, spread, vol_ratio, vol_climax_down, idx, cfg):
    """Check if bar at idx is a Selling Climax."""
    if not vol_climax_down[idx]:
        return False

    # widest spread in lookback
    start = max(0, idx - cfg.climax_spread_lookback)
    max_spread = np.nanmax(spread[start:idx+1])
    if spread[idx] < max_spread * 0.8:
        return False

    return True


def find_buying_climax(high, low, close, volume, spread, vol_ratio, vol_climax_up, idx, cfg):
    """Check if bar at idx is a Buying Climax."""
    if not vol_climax_up[idx]:
        return False

    start = max(0, idx - cfg.climax_spread_lookback)
    max_spread = np.nanmax(spread[start:idx+1])
    if spread[idx] < max_spread * 0.8:
        return False

    return True


def find_automatic_rally(close, volume, vol_ratio, sc_idx, n, cfg):
    """Find Automatic Rally after Selling Climax. Returns AR index or -1."""
    end = min(sc_idx + cfg.ar_max_bars + 1, n)
    sc_close = close[sc_idx]

    best_ar_idx = -1
    best_high = sc_close
    for i in range(sc_idx + 1, end):
        if close[i] > best_high:
            best_high = close[i]
            best_ar_idx = i
        # AR typically has decreasing volume
        if vol_ratio[i] < 1.5 and close[i] > sc_close * 1.02:
            if best_ar_idx > 0:
                return best_ar_idx

    return best_ar_idx


def find_automatic_reaction(close, volume, vol_ratio, bc_idx, n, cfg):
    """Find Automatic Reaction after Buying Climax. Returns AR index or -1."""
    end = min(bc_idx + cfg.ar_max_bars + 1, n)
    bc_close = close[bc_idx]

    best_ar_idx = -1
    best_low = bc_close
    for i in range(bc_idx + 1, end):
        if close[i] < best_low:
            best_low = close[i]
            best_ar_idx = i
        if vol_ratio[i] < 1.5 and close[i] < bc_close * 0.98:
            if best_ar_idx > 0:
                return best_ar_idx

    return best_ar_idx


def label_tr_phases(data, tr_support, tr_resistance, climax_idx, ar_idx, n, cfg, tr_type):
    """
    Label Wyckoff phases A through E for a Trading Range.
    """
    phases = []
    close = data["close"]
    low = data["low"]
    high = data["high"]
    vol_ratio = data["vol_ratio"]
    close_pos = data["close_position"]
    tol = cfg.sr_tolerance

    if tr_type == "accum":
        phase_a, phase_b, phase_c, phase_d, phase_e = (
            WyckoffPhase.accum_A, WyckoffPhase.accum_B, WyckoffPhase.accum_C,
            WyckoffPhase.accum_D, WyckoffPhase.accum_E,
        )
        c_signal = data["spring_signal"]
        d_signal = data["sos_signal"]
    else:
        phase_a, phase_b, phase_c, phase_d, phase_e = (
            WyckoffPhase.distrib_A, WyckoffPhase.distrib_B, WyckoffPhase.distrib_C,
            WyckoffPhase.distrib_D, WyckoffPhase.distrib_E,
        )
        c_signal = data["upthrust_signal"]
        d_signal = data["sow_signal"]

    # phase A: climax to AR
    phases.append((climax_idx, ar_idx, phase_a, 0.8))

    # phase B: oscillation within TR
    phase_b_start = ar_idx + 1
    phase_b_end = min(phase_b_start + cfg.phase_b_max_bars, n - 1)

    # find phase C event (spring / UTAD)
    c_event_idx = -1
    for i in range(phase_b_start, phase_b_end):
        if c_signal[i]:
            c_event_idx = i
            break
        if tr_type == "accum":
            # low volume test near support
            if low[i] < tr_support * (1 - tol) and vol_ratio[i] < cfg.spring_volume_ratio:
                if close_pos[i] > 0.5 and close[i] > tr_support:
                    c_event_idx = i
                    break
        else:
            # high breaks above resistance
            if high[i] > tr_resistance * (1 + tol) and close[i] < tr_resistance:
                if close_pos[i] < 0.5:
                    c_event_idx = i
                    break

    if c_event_idx > 0:
        actual_b_end = max(phase_b_start, c_event_idx - 1)
        if actual_b_end >= phase_b_start:
            phases.append((phase_b_start, actual_b_end, phase_b, 0.7))

        # phase C: spring/UTAD event
        c_end = min(c_event_idx + cfg.spring_recovery_bars, n - 1)
        phases.append((c_event_idx, c_end, phase_c, 0.75))

        # phase D: SOS/SOW + LPS/LPSY
        phase_d_start = c_end + 1
        phase_d_end = min(phase_d_start + 30, n - 1)

        breakout_idx = -1
        for i in range(phase_d_start, phase_d_end):
            if tr_type == "accum":
                if d_signal[i] or (close[i] > tr_resistance * (1 + tol) and vol_ratio[i] > cfg.sos_volume_ratio):
                    breakout_idx = i
                    break
            else:
                if d_signal[i] or (close[i] < tr_support * (1 - tol) and vol_ratio[i] > cfg.sos_volume_ratio):
                    breakout_idx = i
                    break

        if breakout_idx > 0:
            phases.append((phase_d_start, breakout_idx, phase_d, 0.7))
            phase_e_start = breakout_idx + 1
            phase_e_end = min(phase_e_start + 60, n - 1)
            if phase_e_start < n:
                phases.append((phase_e_start, phase_e_end, phase_e, 0.6))
        else:
            if phase_d_start < n:
                phases.append((phase_d_start, phase_d_end, phase_d, 0.5))
    else:
        phases.append((phase_b_start, phase_b_end, phase_b, 0.5))

    return phases


def wyckoff_state_machine(pdf: pd.DataFrame) -> pd.DataFrame:
    """
    Full Wyckoff labeler for one symbol.
    """
    pdf = pdf.sort_values("date").reset_index(drop=True)
    n = len(pdf)

    if pdf.empty:
        return pd.DataFrame(columns=["symbol", "date", "phase_label", "phase_name",
                                      "phase_confidence", "tr_support", "tr_resistance", "tr_id"])

    if n < 60:
        # sett accum_B as default if no enough data
        return pd.DataFrame({
            "symbol": pdf["symbol"],
            "date": pdf["date"],
            "phase_label": [WyckoffPhase.accum_B] * n,
            "phase_name": [WyckoffPhase.accum_B.name] * n,
            "phase_confidence": [0.3] * n,
            "tr_support": [np.nan] * n,
            "tr_resistance": [np.nan] * n,
            "tr_id": [0] * n,
        })

    close = pdf["close"].values
    high = pdf["high"].values
    low = pdf["low"].values
    volume = pdf["volume"].astype(float).values
    spread = pdf["spread"].values if "spread" in pdf.columns else high - low
    sma_20 = pdf["sma_20"].values if "sma_20" in pdf.columns else np.full(n, np.nan)
    sma_50 = pdf["sma_50"].values if "sma_50" in pdf.columns else np.full(n, np.nan)

    vol_sma_20 = pd.Series(volume).rolling(20, min_periods=1).mean().values
    vol_ratio = np.where(vol_sma_20 > 0, volume / vol_sma_20, 1.0)

    vol_climax_up = pdf["volume_climax_up"].values if "volume_climax_up" in pdf.columns else np.zeros(n, dtype=bool)
    vol_climax_down = pdf["volume_climax_down"].values if "volume_climax_down" in pdf.columns else np.zeros(n, dtype=bool)
    spring_signal = pdf["spring_signal"].values if "spring_signal" in pdf.columns else np.zeros(n, dtype=bool)
    upthrust_signal = pdf["upthrust_signal"].values if "upthrust_signal" in pdf.columns else np.zeros(n, dtype=bool)
    sos_signal = pdf["sos_signal"].values if "sos_signal" in pdf.columns else np.zeros(n, dtype=bool)
    sow_signal = pdf["sow_signal"].values if "sow_signal" in pdf.columns else np.zeros(n, dtype=bool)
    close_position = pdf["close_position"].values if "close_position" in pdf.columns else np.full(n, 0.5)

    data = {
        "close": close, "high": high, "low": low, "volume": volume,
        "spread": spread, "vol_ratio": vol_ratio,
        "close_position": close_position,
        "spring_signal": spring_signal, "upthrust_signal": upthrust_signal,
        "sos_signal": sos_signal, "sow_signal": sow_signal,
    }

    # initialize labels: default accum_E (markup) / distrib_E (markdown) based on trend
    labels = np.full(n, -1, dtype=int)
    confidence = np.full(n, 0.0)
    tr_support_arr = np.full(n, np.nan)
    tr_resistance_arr = np.full(n, np.nan)
    tr_id_arr = np.zeros(n, dtype=int)

    # detect trading ranges
    trading_ranges = []  # list type, sc/bc_idx, ar_idx, support, resistance
    tr_counter = 0

    i = cfg.prior_trend_lookback
    while i < n - cfg.ar_max_bars:
        # detect Selling Climax 
        is_downtrend, _ = detect_prior_trend(close, sma_20, sma_50, i, cfg, "down")
        if is_downtrend and find_selling_climax(high, low, close, volume, spread, vol_ratio, vol_climax_down, i, cfg):
            ar_idx = find_automatic_rally(close, volume, vol_ratio, i, n, cfg)
            if ar_idx > 0:
                support = np.min(low[i:ar_idx+1])
                resistance = np.max(high[i:ar_idx+1])
                tr_counter += 1
                trading_ranges.append(("accum", i, ar_idx, support, resistance, tr_counter))
                i = ar_idx + cfg.phase_b_min_bars
                continue

        # detect Buying Climax 
        is_uptrend, _ = detect_prior_trend(close, sma_20, sma_50, i, cfg, "up")
        if is_uptrend and find_buying_climax(high, low, close, volume, spread, vol_ratio, vol_climax_up, i, cfg):
            ar_idx = find_automatic_reaction(close, volume, vol_ratio, i, n, cfg)
            if ar_idx > 0:
                resistance = np.max(high[i:ar_idx+1])
                support = np.min(low[i:ar_idx+1])
                tr_counter += 1
                trading_ranges.append(("distrib", i, ar_idx, support, resistance, tr_counter))
                i = ar_idx + cfg.phase_b_min_bars
                continue

        i += 1

    # label each TR 
    all_phases = []

    for tr_type, climax_idx, ar_idx, support, resistance, tr_id in trading_ranges:
        phases = label_tr_phases(data, support, resistance, climax_idx, ar_idx, n, cfg, tr_type)

        for start, end, phase, conf in phases:
            for j in range(start, min(end + 1, n)):
                if labels[j] == -1 or conf > confidence[j]:
                    labels[j] = int(phase)
                    confidence[j] = conf
                    tr_support_arr[j] = support
                    tr_resistance_arr[j] = resistance
                    tr_id_arr[j] = tr_id

    # fill unlabeled bars with trend-based labels 
    for i in range(n):
        if labels[i] == -1:
            if not np.isnan(sma_20[i]) and not np.isnan(sma_50[i]):
                if sma_20[i] > sma_50[i]:
                    labels[i] = int(WyckoffPhase.accum_E)  # markup
                    confidence[i] = 0.4
                else:
                    labels[i] = int(WyckoffPhase.distrib_E)  # markdown
                    confidence[i] = 0.4
            else:
                # neutral phase B 
                labels[i] = int(WyckoffPhase.accum_B)
                confidence[i] = cfg.min_confidence

    # phase names
    Phase_names = [phase_names.get(l, "UNKNOWN") for l in labels]

    return pd.DataFrame({
        "symbol": pdf["symbol"],
        "date": pdf["date"],
        "phase_label": labels.astype(int),
        "phase_name": Phase_names,
        "phase_confidence": confidence,
        "tr_support": tr_support_arr,
        "tr_resistance": tr_resistance_arr,
        "tr_id": tr_id_arr.astype(int),
    })


#### 3. Cross-split leakage prevent

In [0]:
def wyckoff_state_machine_split_aware(pdf: pd.DataFrame) -> pd.DataFrame:
    """Run wyckoff_state_machine three times with progressively expanding context."""
    pdf = pdf.sort_values("date").reset_index(drop=True)
    if pdf.empty:
        return wyckoff_state_machine(pdf)

    dates = pd.to_datetime(pdf["date"])
    train_end_dt = pd.Timestamp(train_end)
    val_end_dt   = pd.Timestamp(val_end)

    out_frames = []

    # Train
    train_ctx = pdf[dates <= train_end_dt]
    if not train_ctx.empty:
        train_out = wyckoff_state_machine(train_ctx)
        train_out_dates = pd.to_datetime(train_out["date"])
        out_frames.append(train_out[train_out_dates <= train_end_dt])

    # Val
    val_ctx = pdf[dates <= val_end_dt]
    if not val_ctx.empty:
        val_out = wyckoff_state_machine(val_ctx)
        val_out_dates = pd.to_datetime(val_out["date"])
        out_frames.append(val_out[(val_out_dates > train_end_dt) & (val_out_dates <= val_end_dt)])

    # Test
    test_out = wyckoff_state_machine(pdf)
    test_out_dates = pd.to_datetime(test_out["date"])
    out_frames.append(test_out[test_out_dates > val_end_dt])

    if not out_frames:
        return wyckoff_state_machine(pdf.iloc[0:0])

    return pd.concat(out_frames, ignore_index=True)

#### 4. Apply state all symbols

In [0]:
input_cols = [
    "symbol", "date", "open", "high", "low", "close", "volume", "spread",
    "sma_20", "sma_50",
    "volume_climax_up", "volume_climax_down",
    "close_position", "spring_signal", "upthrust_signal",
    "sos_signal", "sow_signal",
]

input_df = joined_df.select(*input_cols)

labels_df = input_df.groupBy("symbol").applyInPandas(wyckoff_state_machine_split_aware, schema=label_schema)

#### 5. Write to Gold

In [0]:
labels_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tables["phase_labels"])
print(f"Successfull written to {tables['phase_labels']}")

#### 6. Verify 

In [0]:
%sql
SELECT
    COUNT(*)                     AS total_rows,
    COUNT(DISTINCT symbol)       AS unique_symbols,
    COUNT(DISTINCT phase_label)  AS distinct_phases
FROM wyckoff_catalog.gold.phase_labels;

In [0]:
%sql
SELECT
    phase_label,
    phase_name,
    COUNT(*)                         AS count,
    ROUND(AVG(phase_confidence), 3)  AS avg_confidence
FROM wyckoff_catalog.gold.phase_labels
GROUP BY phase_label, phase_name
ORDER BY phase_label;

In [0]:
%sql
SELECT *
FROM wyckoff_catalog.gold.phase_labels
WHERE symbol = 'HPG'
ORDER BY date DESC
LIMIT 50;